In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.types import DateTime, Integer, String, Float
import urllib.parse
import logging
import glob
import os
import sys
# ── Logging Setup ──────────────────────────────────────────────────
def setup_logging(log_file="Financial_Performence_Analysis_pipeline.txt"):
    logger = logging.getLogger("FinancialPipeline")
    logger.setLevel(logging.DEBUG)

    formatter = logging.Formatter(
        fmt="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )

    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.INFO)
    console_handler.setFormatter(formatter)

    file_handler = logging.FileHandler(log_file)
    file_handler.setLevel(logging.DEBUG)
    file_handler.setFormatter(formatter)

    logger.addHandler(console_handler)
    logger.addHandler(file_handler)

    return logger


# ── Stage 1: Ingestion ─────────────────────────────────────────────
def ingest(folder_path, logger):
    logger.info("Stage 1 | Ingestion started")
    logger.debug(f"Reading CSVs from: {folder_path}")

    if not os.path.exists(folder_path):
        logger.error(f"Folder not found: {folder_path}")
        raise FileNotFoundError(f"Folder not found: {folder_path}")

    dataframes = {}
    files = glob.glob(os.path.join(folder_path, "*.csv"))

    if not files:
        logger.warning("No CSV files found in the folder")
        return dataframes

    for file in files:
        name = os.path.basename(file).replace(".csv", "")
        try:
            dataframes[name] = pd.read_csv(file)
            logger.debug(f"  Loaded '{name}' → {dataframes[name].shape[0]} rows, {dataframes[name].shape[1]} cols")
        except Exception as e:
            logger.error(f"  Failed to read '{name}': {e}")
            raise

    logger.info(f"Stage 1 | Ingestion complete — {len(dataframes)} files loaded: {list(dataframes.keys())}")
    return dataframes


# ── Stage 2: Clean Budget ──────────────────────────────────────────
def clean_budget(df, logger):
    logger.info("Stage 2 | Cleaning Budget")
    df = df.copy()

    before = df.shape[0]
    df["date"] = pd.to_datetime(df[["year", "month"]].assign(day=1))
    logger.debug(f"  Created 'date' column from year + month")

    nulls = df.isnull().sum().sum()
    logger.debug(f"  Null values after cleaning: {nulls}")
    logger.info(f"  Budget ready → {df.shape[0]} rows (no rows dropped from {before})")
    return df


# ── Stage 3: Clean Customers ───────────────────────────────────────
def clean_customers(df, logger):
    logger.info("Stage 3 | Cleaning Customers")
    df = df.copy()

    before = df.shape[0]
    df["join_date"] = pd.to_datetime(df["join_date"])
    logger.debug(f"  Converted 'join_date' to datetime")

    nulls = df.isnull().sum().sum()
    logger.debug(f"  Null values after cleaning: {nulls}")
    logger.info(f"  Customers ready → {df.shape[0]} rows (no rows dropped from {before})")
    return df


# ── Stage 4: Clean Financial Transactions ─────────────────────────
def clean_transactions(df, logger):
    logger.info("Stage 4 | Cleaning Financial Transactions")
    df = df.copy()

    before = df.shape[0]
    df["transaction_date"] = pd.to_datetime(df["transaction_date"])
    logger.debug(f"  Converted 'transaction_date' to datetime")

    df["amount_abs"] = df["amount"].abs()
    logger.debug(f"  Created 'amount_abs' column")

    null_counts = df.isnull().sum()
    null_cols = null_counts[null_counts > 0]
    if not null_cols.empty:
        logger.warning(f"  Nulls found:\n{null_cols}")
    else:
        logger.debug("  No null values found")

    logger.info(f"  Transactions ready → {df.shape[0]} rows (no rows dropped from {before})")
    return df


# ── Stage 5: Clean Employees ──────────────────────────────────────
def clean_employees(df, logger):
    logger.info("Stage 5 | Cleaning Employees")
    df = df.copy()

    before = df.shape[0]
    df["join_date"] = pd.to_datetime(df["join_date"])
    logger.debug(f"  Converted 'join_date' to datetime")

    nulls = df.isnull().sum().sum()
    logger.debug(f"  Null values after cleaning: {nulls}")
    logger.info(f"  Employees ready → {df.shape[0]} rows (no rows dropped from {before})")
    return df


# ── Stage 6: Vendors (pass-through) ───────────────────────────────
def clean_vendors(df, logger):
    logger.info("Stage 6 | Vendors — no cleaning required")
    logger.debug(f"  Vendors shape: {df.shape}")
    return df.copy()


# ── Stage 7: Database Engine ───────────────────────────────────────
def create_db_engine(server_name, database_name, logger):
    logger.info("Stage 7 | Creating database engine...")
    try:
        params = urllib.parse.quote_plus(
            f'DRIVER={{ODBC Driver 18 for SQL Server}};'
            f'SERVER={server_name};'
            f'DATABASE={database_name};'
            f'Trusted_Connection=yes;'
            f'Encrypt=yes;'
            f'TrustServerCertificate=yes;'
        )
        engine = create_engine(
            f"mssql+pyodbc:///?odbc_connect={params}",
            fast_executemany=True
        )
        logger.info("  Database engine created successfully.")
        return engine
    except Exception as e:
        logger.error(f"  Failed to create engine: {e}")
        raise


# ── Stage 8: Smart Upload ──────────────────────────────────────────
def upload_to_sql(df, table_name, engine, unique_keys, dtype_map, logger):
    logger.info(f"  Uploading '{table_name}' — {len(df)} records...")

    try:
        inspector    = inspect(engine)
        table_exists = inspector.has_table(table_name)

        # CASE 1 — Fresh upload
        if not table_exists:
            logger.info(f"  Table '{table_name}' not found — fresh upload...")
            df["uploaded_at"] = pd.Timestamp.now()
            df.to_sql(
                table_name,
                engine,
                if_exists="replace",
                index=False,
                chunksize=10000,
                dtype=dtype_map
            )
            logger.info(f"  Fresh upload complete — {len(df)} records inserted.")

        # CASE 2 — Smart insert (no duplicates)
        else:
            logger.info(f"  Table '{table_name}' exists — checking for duplicates...")

            existing_df = pd.read_sql(
                f"SELECT {', '.join(unique_keys)} FROM {table_name}",
                engine
            )
            logger.info(f"  Existing records in DB: {len(existing_df)}")

            merged   = df.merge(existing_df, on=unique_keys, how="left", indicator=True)
            new_rows = merged[merged["_merge"] == "left_only"].drop(columns="_merge")

            if len(new_rows) == 0:
                logger.info(f"  No new records for '{table_name}' — upload skipped.")
                return

            new_rows["uploaded_at"] = pd.Timestamp.now()
            new_rows.to_sql(
                table_name,
                engine,
                if_exists="append",
                index=False,
                chunksize=10000,
                dtype=dtype_map
            )
            logger.info(
                f"  Smart upload complete — "
                f"{len(new_rows)} new records inserted, "
                f"{len(df) - len(new_rows)} duplicates skipped."
            )

    except Exception as e:
        logger.error(f"  Upload failed for '{table_name}': {e}")
        raise


# ── dtype maps ─────────────────────────────────────────────────────
DTYPE_MAPS = {
    "Budget": {
        "year":             Integer(),
        "month":            Integer(),
        "business_unit":    String(50),
        "budgeted_revenue": Integer(),
        "budgeted_expense": Integer(),
        "date":             DateTime()
    },
    "Customers": {
        "customer_id":   String(50),
        "customer_name": String(200),
        "segment":       String(50),
        "join_date":     DateTime(),
        "region":        String(50),
        "status":        String(20)
    },
    "Financial_Transactions": {
        "transaction_id":   String(50),
        "transaction_date": DateTime(),
        "amount":           Float(),
        "account_type":     String(50),
        "category":         String(50),
        "business_unit":    String(50),
        "region":           String(50),
        "customer_id":      String(50),
        "vendor_id":        String(50),
        "description":      String(500),
        "amount_abs":       Float()
    },
    "Employees": {
        "employee_id":     String(50),
        "employee_name":   String(200),
        "business_unit":   String(50),
        "join_date":       DateTime(),
        "status":          String(20),
        "region":          String(50),
        "cost_to_company": Integer()
    },
    "Vendors": {
        "vendor_id":   String(50),
        "vendor_name": String(200),
        "category":    String(50),
        "region":      String(50),
        "active":      String(5)
    }
}

# ── unique keys ────────────────────────────────────────────────────
UNIQUE_KEYS = {
    "Budget":                 ["year", "month", "business_unit"],
    "Customers":              ["customer_id"],
    "Financial_Transactions": ["transaction_id"],
    "Employees":              ["employee_id"],
    "Vendors":                ["vendor_id"]
}


# ── Master Pipeline ────────────────────────────────────────────────
def run_pipeline(folder_path, server_name, database_name, log_file="Financial_Performence_Analysis_pipeline.txt"):
    logger = setup_logging(log_file)
    logger.info("=" * 60)
    logger.info("Financial Performance Pipeline — START")
    logger.info("=" * 60)

    try:
        # Cleaning stages
        raw          = ingest(folder_path, logger)
        budget       = clean_budget(raw["Budget"], logger)
        customers    = clean_customers(raw["Customers"], logger)
        transactions = clean_transactions(raw["Financial_Transactions"], logger)
        employees    = clean_employees(raw["Headcount"], logger)
        vendors      = clean_vendors(raw["Vendors"], logger)

        # Database stage
        engine = create_db_engine(server_name, database_name, logger)

        logger.info("Stage 8 | Uploading all tables to SQL Server")
        upload_to_sql(budget,       "Budget",                 engine, UNIQUE_KEYS["Budget"],                 DTYPE_MAPS["Budget"],                 logger)
        upload_to_sql(customers,    "Customers",              engine, UNIQUE_KEYS["Customers"],              DTYPE_MAPS["Customers"],              logger)
        upload_to_sql(transactions, "Financial_Transactions", engine, UNIQUE_KEYS["Financial_Transactions"], DTYPE_MAPS["Financial_Transactions"], logger)
        upload_to_sql(employees,    "Employees",              engine, UNIQUE_KEYS["Employees"],              DTYPE_MAPS["Employees"],              logger)
        upload_to_sql(vendors,      "Vendors",                engine, UNIQUE_KEYS["Vendors"],                DTYPE_MAPS["Vendors"],                logger)

        logger.info("=" * 60)
        logger.info("Financial Performance Pipeline — COMPLETE")
        logger.info("=" * 60)

    except Exception as e:
        logger.critical(f"Pipeline FAILED: {e}", exc_info=True)
        raise


# ── Run ────────────────────────────────────────────────────────────
folder_path   = r"D:\DATA SETS (Project)\Capston Project Data\Financial Performence Analysis"
server_name   = r"LAPTOP-2VUPGIQA\SQLEXPRESS"
database_name = "FinancialPerformance_DB"

run_pipeline(folder_path, server_name, database_name)

2026-05-26 07:31:42 | INFO     | FinancialPipeline | ============================================================
2026-05-26 07:31:42 | INFO     | FinancialPipeline | Financial Performance Pipeline — START
2026-05-26 07:31:42 | INFO     | FinancialPipeline | ============================================================
2026-05-26 07:31:42 | INFO     | FinancialPipeline | Stage 1 | Ingestion started
2026-05-26 07:31:42 | INFO     | FinancialPipeline | Stage 1 | Ingestion complete — 5 files loaded: ['Budget', 'Customers', 'Financial_Transactions', 'Headcount', 'Vendors']
2026-05-26 07:31:42 | INFO     | FinancialPipeline | Stage 2 | Cleaning Budget
2026-05-26 07:31:42 | INFO     | FinancialPipeline |   Budget ready → 72 rows (no rows dropped from 72)
2026-05-26 07:31:42 | INFO     | FinancialPipeline | Stage 3 | Cleaning Customers
2026-05-26 07:31:42 | INFO     | FinancialPipeline |   Customers ready → 400 rows (no rows dropped from 400)
2026-05-26 07:31:42 | INFO     | FinancialPipeline 

--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\user\anaconda3\Lib\logging\__init__.py", line 1154, in emit
    stream.write(msg + self.terminator)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\user\anaconda3\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\u2192' in position 71: character maps to <undefined>
Call stack:
  File "C:\Users\user\anaconda3\Lib\runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\user\anaconda3\Lib\runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "C:\Users\user\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\user\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in

2026-05-26 07:31:42 | INFO     | FinancialPipeline |   Database engine created successfully.
2026-05-26 07:31:42 | INFO     | FinancialPipeline | Stage 8 | Uploading all tables to SQL Server
2026-05-26 07:31:42 | INFO     | FinancialPipeline |   Uploading 'Budget' — 72 records...
2026-05-26 07:31:43 | INFO     | FinancialPipeline |   Table 'Budget' exists — checking for duplicates...
2026-05-26 07:31:43 | INFO     | FinancialPipeline |   Existing records in DB: 72
2026-05-26 07:31:43 | INFO     | FinancialPipeline |   No new records for 'Budget' — upload skipped.


C:\Users\user\AppData\Local\Temp\ipykernel_20504\2652327373.py:168: SAWarning: Unrecognized server version info '17.0.1115.1'.  Some SQL Server features may not function properly.
  inspector    = inspect(engine)


2026-05-26 07:31:43 | INFO     | FinancialPipeline |   Uploading 'Customers' — 400 records...
2026-05-26 07:31:43 | INFO     | FinancialPipeline |   Table 'Customers' exists — checking for duplicates...
2026-05-26 07:31:43 | INFO     | FinancialPipeline |   Existing records in DB: 400
2026-05-26 07:31:43 | INFO     | FinancialPipeline |   No new records for 'Customers' — upload skipped.
2026-05-26 07:31:43 | INFO     | FinancialPipeline |   Uploading 'Financial_Transactions' — 10400 records...
2026-05-26 07:31:43 | INFO     | FinancialPipeline |   Table 'Financial_Transactions' exists — checking for duplicates...
2026-05-26 07:31:43 | INFO     | FinancialPipeline |   Existing records in DB: 10400
2026-05-26 07:31:43 | INFO     | FinancialPipeline |   No new records for 'Financial_Transactions' — upload skipped.
2026-05-26 07:31:43 | INFO     | FinancialPipeline |   Uploading 'Employees' — 200 records...
2026-05-26 07:31:43 | INFO     | FinancialPipeline |   Table 'Employees' exists — c